In [1]:
import pandas as pd

df = pd.read_csv("Job Datsset.csv")

print(df.head())

   User_ID  Job_ID                         User_Skills  \
0        1      16       Python, C++, Machine Learning   
1        2      30            AI, Python, Data Science   
2        3     319       CSS, Python, Machine Learning   
3        4     399          SQL, Machine Learning, C++   
4        5     405  Machine Learning, HTML, JavaScript   

                                   Job_Requirements  Match_Score  Recommended  
0            SQL, CSS, AI, JavaScript, Data Science     0.620421            0  
1                AI, Data Science, SQL, Python, CSS     0.823451            1  
2                                   SQL, AI, Python     0.703830            0  
3  Java, AI, Python, Data Science, Machine Learning     0.224724            0  
4                             Machine Learning, C++     0.296453            0  


In [2]:
import spacy
from spacy.matcher import PhraseMatcher
#Loads the English language model so spaCy can process text.
nlp = spacy.load("en_core_web_sm")

In [3]:
import sys
print(sys.executable)

c:\Users\Hari\AppData\Local\Programs\Python\Python311\python.exe


In [4]:
!{sys.executable} -m pip install spacy
!{sys.executable} -m spacy download en_core_web_sm


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     ---- ----------------------------------- 1.6/12.8 MB 4.9 MB/s eta 0:00:03
     ---------- ----------------------------- 3.4/12.8 MB 6.7 MB/s eta 0:00:02
     ------------- -------------------------- 4.2/12.8 MB 6.3 MB/s eta 0:00:02
     -------------- ------------------------- 4.7/12.8 MB 5.4 MB/s eta 0:00:02
     ------------------ --------------------- 6.0/12.8 MB 5.1 MB/s eta 0:00:02
     ------------------------ --------------- 7.9/12.8 MB 5.9 MB/s eta 0:00:01
     ---------------------------- ----------- 9.2/12.8 MB 6.0 MB/s eta 0:00:01
     ------------------------------ --------- 9.7/12.8 MB 5.8 MB/s eta 0:00:01
     ------------------------------- -------- 10.2/12.8 MB 5.3 MB/s eta 0:00:01
     -------------------------------- ------- 10.5/12.8 MB 4.9 MB/s eta 


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
df = pd.read_csv("job Datsset.csv", header=None)
#EDA
df.head()

,0,1,2,3,4,5
0,User_ID,Job_ID,User_Skills,Job_Requirements,Match_Score,Recommended
1,1,16,"Python, C++, Machine Learning","SQL, CSS, AI, JavaScript, Data Science",0.6204206284876945,0
2,2,30,"AI, Python, Data Science","AI, Data Science, SQL, Python, CSS",0.8234510612396936,1
3,3,319,"CSS, Python, Machine Learning","SQL, AI, Python",0.7038296838133097,0
4,4,399,"SQL, Machine Learning, C++","Java, AI, Python, Data Science, Machine Learning",0.22472441131493304,0


In [6]:
print("Shape of Dataset:", df.shape)
print("Number of Rows:", df.shape[0])
print("Number of Columns:", df.shape[1])

Shape of Dataset: (100001, 6)
Number of Rows: 100001
Number of Columns: 6


In [7]:
print(df.columns)

Index([0, 1, 2, 3, 4, 5], dtype='int64')


In [8]:
#Create Skill Patterns
skills = [
    "python",
    "sql",
    "machine learning",
    "java",
    "data analysis",
    "tensorflow",
    "deep learning"
]

matcher = PhraseMatcher(nlp.vocab)

patterns = [nlp(skill) for skill in skills]

matcher.add("SKILLS", patterns)

In [9]:
#Extract Skills from User Profile and Job Description
def extract_skills(text):
    
    doc = nlp(str(text).lower())

    matches = matcher(doc)

    found_skills = []

    for match_id, start, end in matches:
        found_skills.append(doc[start:end].text)

    return list(set(found_skills))

In [10]:
df = pd.read_csv("Job Datsset.csv", delimiter=",")
print(df.columns.tolist())

['User_ID', 'Job_ID', 'User_Skills', 'Job_Requirements', 'Match_Score', 'Recommended']


In [11]:
print(df.head())
print(df.shape)

   User_ID  Job_ID                         User_Skills  \
0        1      16       Python, C++, Machine Learning   
1        2      30            AI, Python, Data Science   
2        3     319       CSS, Python, Machine Learning   
3        4     399          SQL, Machine Learning, C++   
4        5     405  Machine Learning, HTML, JavaScript   

                                   Job_Requirements  Match_Score  Recommended  
0            SQL, CSS, AI, JavaScript, Data Science     0.620421            0  
1                AI, Data Science, SQL, Python, CSS     0.823451            1  
2                                   SQL, AI, Python     0.703830            0  
3  Java, AI, Python, Data Science, Machine Learning     0.224724            0  
4                             Machine Learning, C++     0.296453            0  
(100000, 6)


In [ ]:
#Apply Skill Extraction
df["Extracted_User_Skills"] = df["User_Skills"].apply(extract_skills)

df["Extracted_Job_Skills"] = df["Job_Requirements"].apply(extract_skills)

In [ ]:
def skill_match(user_skills, job_skills):

    user_set = set(user_skills)
    job_set = set(job_skills)

    if len(job_set) == 0:
        return 0

    return len(user_set.intersection(job_set)) / len(job_set)

df["Similarity"] = df.apply(
    lambda x: skill_match(
        x["Extracted_User_Skills"],
        x["Extracted_Job_Skills"]
    ),
    axis=1
)

In [ ]:
recommendations = df.sort_values(
    by="Similarity",
    ascending=False
)

print(
    recommendations[
        ["Job_ID", "Similarity"]
    ].head(10)
)